In [ ]:
# Import necessary library
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.prompts import PromptTemplate

from langchain_community.chains import RetrievalQA

In [2]:
# Read pdfs from the folder
loader = PyPDFDirectoryLoader("./us_census")

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

final_documents = text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.\n1 Public policy changes included \nthe renewal of the Public Health Emergency, w

In [3]:
len(final_documents)

316

In [6]:
# Embeddings using Huggingface
huggingface_embeddings = HuggingFaceBgeEmbeddings(
    model_name  = "BAAI/bge-large-en-v1.5",
    model_kwargs = {'device':'cpu'},
    encode_kwargs = {'normalize_embeddings':True}
)

/Users/kirtiverma/Projects/Practice2/Langchain/genai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2403.87it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
import numpy as np
np.array(huggingface_embeddings.embed_query(final_documents[0].page_content))
print(np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)).shape)

(1024,)


In [9]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(final_documents[:120],huggingface_embeddings)


In [10]:
# Query using similarity Search
query = "What is health insurance coverage?"
relevant_doc = vector_store.similarity_search(query)
print(relevant_doc[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [11]:
retriever = vector_store.as_retriever(search_type = "similarity",search_kwargs = {"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x33bbde910> search_kwargs={'k': 3}


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [26]:
from langchain_community.llms import Ollama

llm= Ollama(model = "llama2")
llm.invoke(query)


/var/folders/f1/562xtxbj6z5gck527v2ky4cw0000gn/T/ipykernel_98243/1616967566.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm= Ollama(model = "llama2")


"\nThe health insurance coverage for a 27-year-old in the United States can vary depending on factors such as their state of residence, employment status, and individual circumstances. Here are some general options for health insurance coverage for a 27-year-old in the US:\n\n1. Affordable Care Act (ACA) Marketplace Plans: As a self-employed individual, your income may be above the threshold for financial assistance through the ACA marketplace. However, you may still be eligible for subsidies based on your income and family size. You can explore options through HealthCare.gov or your state's marketplace.\n2. Group Health Insurance: If you work for an employer that offers health insurance, you may be eligible for coverage through their group plan. This can be a good option if the plan is affordable and provides adequate coverage.\n3. Short-Term Health Insurance: These plans provide temporary coverage for a limited period (usually up to 12 months) and are often less expensive than compre

In [28]:
#Hugging Face models can be run locally through the HuggingFacePipeline class.
from langchain_huggingface import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 100},
    
)

llm = hf 
llm.invoke(query)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2605.75it/s, Materializing param=model.norm.weight]                              
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'What is the health insurance coverage?\n\nI get asked this question a lot so I thought I’d write a blog about it.  This isn’t just for expats, but for all Americans.\n\nFirst of all, you need to know that it’s an individual mandate.  This means that you have to buy it.  It doesn’t matter if you are rich or poor, you must buy it.  If you don’t have it you will get a tax penalty.  This'

In [29]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [30]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [33]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create RAG chain
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | hf
    | StrOutputParser()
)

# Invoke
response = rag_chain.invoke("What is the health insurance coverage?")
print(response)


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

[Document(id='543b7ea9-d0e4-494d-8fea-c087fb5930db', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2'}, page_content='2 U.S. Census Bureau\nWHAT IS HEALTH INSURANCE COVERAGE?\nThis brief presents state-level estimates of health insurance coverage \nusing data from the American Community Survey (ACS). The  \nU.S. Census Bureau conducts the ACS throughout the year; the \nsurvey asks respondents to report their coverage at the time of \ninterview. The resulting measure of health insurance coverage, 